# Gate: SGLang + OLMoE on Kaggle 2×T4

Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Add your HF token as a Kaggle secret named `HF_TOKEN`.

Each step prints **PASS** or **FAIL** plus the key numbers.
Time box: 4 hours. If the server never comes up, the fallbacks are in step 4.

In [ ]:
# 1. GPU + driver check -> decides which SGLang to install
import subprocess, re
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(smi)
drv = re.search(r"Driver Version:\s*([0-9.]+)", smi)
DRIVER = drv.group(1) if drv else None
DRIVER_MAJOR = int(DRIVER.split(".")[0]) if DRIVER else 0
# Current SGLang needs CUDA 13 (driver >= 580). Older drivers -> last CUDA 12 build (v0.5.19, cu129).
SGLANG_SPEC = "sglang" if DRIVER_MAJOR >= 580 else "sglang==0.5.19"
print(f"driver={DRIVER}  ->  will install: {SGLANG_SPEC}")
print("PASS" if DRIVER else "FAIL: no driver found (is the GPU accelerator on?)")

In [ ]:
# 2. Install. Verified on Kaggle 2xT4 (driver 580, Python 3.13) 2026-10-14: sglang 0.5.21 + torch 2.13.0.
#    outlines_core has no Python 3.13 wheel, so it builds from source and needs Rust.
#    Ignore pip's "incompatible" warnings about RAPIDS/cudf/google packages: unused here.
#    Run All from a fresh session works: nothing above imports torch. If torch was already imported in this
#    session (e.g. you reran step 3 first), do Run -> Restart and continue from step 3.
import os, sys
print("python", sys.version.split()[0])
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal 2>&1 | tail -1
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
SPEC = SGLANG_SPEC if SGLANG_SPEC != "sglang" else "sglang==0.5.21"
!pip install "{SPEC}" "pybase64" 2>&1 | grep -iE "^error|successfully installed" | tail -5
# Kaggle preinstalls a cu128 torchaudio; transformers imports it and it crashes against cu130 torch. Unused here.
!pip uninstall -y -q torchaudio
# No prebuilt kernels for sm75: FlashInfer JIT-compiles them (e.g. RMSNorm) and links with -lcuda, but Kaggle
# only has the driver's libcuda.so.1 -> "ld: cannot find -lcuda". Give the linker the unversioned name.
# FlashInfer JIT-compiles sm75 kernels and links with -lcuda. The driver's unversioned libcuda.so lives in
# /usr/local/nvidia/lib64 (not on the linker path, not in ldconfig; /usr/local/cuda is read-only), so add it to
# LIBRARY_PATH; the server subprocess inherits os.environ. Do NOT use the cuda-12.8/compat (570) or ~/.julia (610)
# copies: they don't match driver 580. Kaggle's system nvcc is CUDA 12.8 while torch is cu130; driver 580 runs both.
import subprocess
NV = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = NV + ":" + os.environ.get("LIBRARY_PATH", "")
print("system nvcc:", subprocess.run(["/usr/local/cuda/bin/nvcc", "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1:])
print("libcuda.so ->", os.path.realpath(f"{NV}/libcuda.so") if os.path.exists(f"{NV}/libcuda.so") else "NOT FOUND: FlashInfer JIT link will fail")


In [ ]:
# 3. Import check + capability
import torch, importlib.metadata as md
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("gpus:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      "capability:", [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())])
ok = True
for pkg in ("sglang", "sglang-kernel", "flashinfer-python"):
    try: print(pkg, md.version(pkg))
    except md.PackageNotFoundError: print(pkg, "NOT INSTALLED"); ok = ok and pkg != "sglang"
try:
    import sglang
    from sglang.srt.state_capturer.routed_experts import extract_routed_experts_from_meta_info
    print("import sglang + routed_experts helper: ok")
except Exception as e:
    ok = False; print("import error:", repr(e))
print("PASS" if ok else "FAIL")

In [ ]:
# 4. Server helpers. Every flag below was checked against SGLang source (arg_groups/fields/exec_.py).
#    Fallbacks if launch fails: (a) drop --tp 2 and use --mem-fraction-static 0.9 on one GPU,
#    (b) MODEL = "Qwen/Qwen1.5-MoE-A2.7B", (c) skip SGLang today and run HF-only (step 7).
import os, sys, time, signal, subprocess, requests
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret (fine for public models)")

MODEL = "allenai/OLMoE-1B-7B-0924"
PORT = 30000
BASE = f"http://127.0.0.1:{PORT}"
proc = None
LOG = "/kaggle/working/server.log"   # /kaggle/working survives as notebook output; plain cwd files vanish with the session

def dump_log():
    import re
    text = open(LOG).read()
    pat = r"error|exception|not supported|unsupported|sm_?75|capability|out of memory|no kernel image|Traceback"
    hits = [l for l in text.splitlines() if re.search(pat, l, re.I)]
    print("=" * 30, f"ERROR LINES ({len(hits)}) from {LOG}", "=" * 30)
    print("\n".join(hits[-60:]))
    print("=" * 30, "LAST 80 LOG LINES", "=" * 30)
    print("\n".join(text.splitlines()[-80:]))

def launch(extra=()):
    global proc
    stop()
    cmd = [sys.executable, "-m", "sglang.launch_server", "--model-path", MODEL, "--port", str(PORT),
           "--dtype", "float16", "--tp", "2",
           "--attention-backend", "triton", "--sampling-backend", "pytorch",
           "--disable-cuda-graph", "--enable-return-routed-experts",
           "--mem-fraction-static", "0.85", *extra]
    print(" ".join(cmd))
    log = open(LOG, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, preexec_fn=os.setsid)
    t0 = time.time()
    while time.time() - t0 < 900:
        if proc.poll() is not None:
            dump_log(); raise RuntimeError("server exited: see the ERROR LINES above")
        try:
            if requests.get(f"{BASE}/health", timeout=2).status_code == 200:
                print(f"server up in {time.time()-t0:.0f}s"); return
        except Exception:
            pass
        time.sleep(5)
    dump_log(); raise TimeoutError("server not healthy after 15 min: see the ERROR LINES above")

def stop():
    global proc
    if proc and proc.poll() is None:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM); proc.wait(timeout=60)
    proc = None

In [ ]:
# 5. Launch + smoke test: 5 prompts, greedy, check text + routed experts decode
import numpy as np, json
from transformers import AutoConfig
from sglang.srt.state_capturer.routed_experts import extract_routed_experts_from_meta_info

hf_cfg = AutoConfig.from_pretrained(MODEL)
N_LAYERS, TOPK = hf_cfg.num_hidden_layers, hf_cfg.num_experts_per_tok
print("layers", N_LAYERS, "experts", hf_cfg.num_experts, "top_k", TOPK)

PROMPTS = [
    "The capital of France is",
    "def fibonacci(n):",
    "Explain why the sky is blue in one sentence.",
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did she sell altogether?",
    "Translate to German: The weather is nice today.",
]

def generate(texts, max_new_tokens=32):
    r = requests.post(f"{BASE}/generate", json={
        "text": texts,
        "sampling_params": {"temperature": 0.0, "max_new_tokens": max_new_tokens},
        "return_routed_experts": True,
    }, timeout=600)
    r.raise_for_status()
    out = r.json()
    return out if isinstance(out, list) else [out]

def routes(item):
    arr = np.asarray(extract_routed_experts_from_meta_info(item))
    return arr.reshape(-1, N_LAYERS, TOPK)

launch()
res = generate(PROMPTS)
bad = 0
for p, item in zip(PROMPTS, res):
    txt = item["text"]
    r = routes(item)
    bad += (not txt.strip()) or ("nan" in txt.lower())
    print(f"{p[:40]!r:44} -> {txt[:60]!r}  routes{tuple(r.shape)}")
print("meta_info keys:", sorted(res[0]["meta_info"].keys()))
print("PASS" if bad == 0 else f"FAIL: {bad} empty/NaN outputs")

In [ ]:
# 6. Batch-size effect, stock vs deterministic mode.
#    Same target prompt alone (bs=1) vs with 7 different neighbours (bs=8). Only PROMPT positions are
#    compared: both runs see identical tokens there (prefill), so a difference is a real route flip, not
#    a knock-on from different generated text. R0 = bs=1 run twice (noise floor). Smoke test, not the census.
from transformers import AutoTokenizer
TARGET = PROMPTS[3]
NEIGHBOURS = [f"Write a short story about a {w}." for w in ["dragon","robot","sailor","chef","pilot","river","comet"]]
N_PROMPT = len(AutoTokenizer.from_pretrained(MODEL)(TARGET).input_ids)

def flip_rate(a, b):
    n = min(N_PROMPT, len(a), len(b))
    return (np.sort(a[:n], -1) != np.sort(b[:n], -1)).any(-1).mean(), n   # top-k SET mismatch per (position, layer)

def run_mode(label):
    solo1 = routes(generate([TARGET], max_new_tokens=1)[0])
    solo2 = routes(generate([TARGET], max_new_tokens=1)[0])
    batched = routes(generate([TARGET] + NEIGHBOURS, max_new_tokens=1)[0])
    r0, n = flip_rate(solo1, solo2)
    rb, _ = flip_rate(solo1, batched)
    print(f"{label:14} R0 bs1 vs bs1: {r0:.4%}   bs1 vs bs8: {rb:.4%}   excess over R0: {rb - r0:+.4%}   ({n} prompt positions x {N_LAYERS} layers)")
    return r0, rb

r0_stock, rb_stock = run_mode("stock:")
launch(["--enable-deterministic-inference"])
r0_det, rb_det = run_mode("deterministic:")
stop()
print("R0 PASS" if r0_stock == 0 and r0_det == 0 else "R0 NONZERO: bs=1 is not even repeatable -> note it, every effect must be reported over this floor")
print("DET PASS" if rb_det == 0 else "DET FAIL: deterministic mode still flips -> note it, this is itself a finding")


In [ ]:
# 7. R1 sanity: SGLang fp16 vs HF fp16 next-token argmax on 20 prompts + overflow check.
#    (fp32 oracle runs later on Modal/laptop: OLMoE fp32 = ~28 GB, too big for Kaggle RAM.)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
model.eval()

absmax, nonfinite = 0.0, 0
def hook(_m, _i, out):
    global absmax, nonfinite
    t = out[0] if isinstance(out, tuple) else out
    absmax = max(absmax, t.detach().abs().float().max().item())
    nonfinite += (~torch.isfinite(t)).sum().item()
hooks = [layer.register_forward_hook(hook) for layer in model.model.layers]

PROMPTS20 = (PROMPTS * 4)[:20]
hf_next = []
with torch.no_grad():
    for p in PROMPTS20:
        ids = tok(p, return_tensors="pt").to(model.device)
        hf_next.append(model(**ids).logits[0, -1].argmax().item())
for h in hooks: h.remove()
del model; torch.cuda.empty_cache()

launch()
sg = generate(PROMPTS20, max_new_tokens=1)
stop()
# compare token ids (decoded text can differ in leading spaces); fall back to text if ids are absent
def sg_first(item):
    ids = item.get("output_ids")
    return ids[0] if ids else item["text"]
hf_cmp = [t if "output_ids" in sg[0] else tok.decode([t]) for t in hf_next]
agree = sum(1 for h, item in zip(hf_cmp, sg) if sg_first(item) == h) / len(hf_cmp)
print("compared by:", "token id" if "output_ids" in sg[0] else "decoded text")
print(f"argmax agreement HF fp16 vs SGLang fp16: {agree:.0%}   residual absmax={absmax:.1f}   non-finite={nonfinite}")
print("PASS" if agree >= 0.95 and nonfinite == 0 else "CHECK: low agreement or overflow -> keep residual stream fp32")

## Done
The numbers from step 6 are a first data point only; the census itself uses teacher forcing.